# 凑出 24：由四个数与运算符组合出一条成立的算式

## 这一章讲什么

这里把 24 点这个游戏交给 Z3 来解，四个数怎样排列、三个运算符怎样挑选、按什么顺序计算，这三件事一旦确定，式子也就定了。

- 四个数本质上是一次重排，其排列同样用约束来写(有重复的数也照样跑得动)
- 计算到一半会出现分数，因此变量必须使用 `Real`(实数)
- 运算符编成 0〜5 的编号，再用 `Implies` 说明“取这个编号时应当如何计算”

同一道题，后面我们还会用 `Function()` 再写一版。

## 问题

给出四个数，再给出 `+`, `-`, `*`, `/` 和括号，请凑出一条等于 `24` 的算式。

拿 `[1, 5, 5, 5]` 来说

```
(5 - (1 / 5)) * 5 = 24
```

上面这条式子就是一个合格的答案。数的排法有 $4! = 24$ 种(有重复的数会少一些)，运算符的选法有 $6^3$ 种(怎么来的，后面再说)。这项工作人手工去找相当累，因此干脆让 Z3 去翻，下面先看数的排列怎样写成约束。

## 用约束表示数的排列

`solve_24` 需要的是“排好序的四个数”，因此排列这件事也必须写成约束，交给求解器去挑选。

`permutation(xs, values)` 给出的约束，作用是让 `xs` 成为 `values` 的一次重排。

1. `xs` 里的每个元素，都必须是 `values` 中出现过的值
2. 每个值 `v` 在 `xs` 里出现的次数，要与其在 `values` 里出现的次数一样

`[1, 5, 5, 5]` 的说法就是 `1` 出现 1 次、`5` 出现 3 次。这样写的好处是即使有多个相同的数也能正确处理，而换成 `Distinct()` 那一路就做不到。

In [1]:
from z3 import *
from helper.z3 import all_solutions


def permutation(variables, values):
    """返回 variables 是 values 的一个排列这一约束"""
    distinct_values = set(values)
    # 1. 每个元素都是 values 中含有的值之一
    exprs = [Or([x == v for v in distinct_values]) for x in variables]
    # 2. 每个值的出现次数与 values 相同
    exprs += [Sum([x == v for x in variables]) == values.count(v) for v in distinct_values]
    return And(exprs)


xs = IntVector('x', 4)
solver = Solver()
solver.add(permutation(xs, [1, 5, 5, 5]))

for m in all_solutions(solver):
    print([m[x] for x in xs])

[5, 5, 5, 1]
[1, 5, 5, 5]
[5, 5, 1, 5]
[5, 1, 5, 5]


`[1, 5, 5, 5]` 的重排只有 $4!/3! = 4$ 种，其原因是三个 5 相同，由于重复的部分要除掉，因此输出也只有 4 行。

## 组装算式

我们把计算过程拉成一条直线，从左往右一步一步计算，每一步只做一次运算。

```
s0 = n0 op0 n1
s1 = s0 op1 n2
s2 = s1 op2 n3      ← 算到 24 就成
```

这个形状的好处是节省：四个数、三个运算符、三个中间结果，一条式子就装下了，不必为括号另想办法，也不会多出任何多余的状态。

`s2` 算出来以后，再倒回去把它拼成原来的算式字符串。

In [2]:
# 运算符编号 → 符号
# 0:+  1:-  2:*  3:/  4:反向的-  5:反向的/
OP_SYMBOLS = {0: '+', 1: '-', 2: '*', 3: '/', 4: '-', 5: '/'}


def output(numbers, ops):
    """从计算步骤（数的顺序和运算符序列）生成算式字符串"""
    def combine(n1, n2, op):
        if op in (4, 5):
            n1, n2 = n2, n1          # 反向的 - 和 / 交换左右再显示
        if isinstance(n1, str):
            n1 = f'({n1})'
        if isinstance(n2, str):
            n2 = f'({n2})'
        return f'{n1} {OP_SYMBOLS[op]} {n2}'

    expr = numbers[0]
    for n2, op in zip(numbers[1:], ops):
        expr = combine(expr, n2, op)
    return expr


print(output([1, 5, 5, 5], [3, 4, 2]))   # (5 - (1 / 5)) * 5
print(output([8, 3, 3, 8], [3, 4, 5]))   # 8 / (3 - (8 / 3))

(5 - (1 / 5)) * 5
8 / (3 - (8 / 3))


运算符编到 0〜5 六个编号，是因为减法与除法的左右交换形式也各占一个编号。

- `-` 要准备 `a - b` 和 `b - a` 两份(不重排数也能两边都试到)
- `/` 同样如此，`a / b` 与 `b / a` 都要

加法与乘法换个顺序结果不变，只有减法与除法在乎谁在左谁在右。编号分开编制，就不必在数的排法上白费力气。

## 用 Z3 求解

这里的关键一步，是把四个数设成 `Real`(实数)。中途会出现 `8 / 3` 这种分数，若用整数除下去就会断在半路，因为整数除法会把小数部分直接丢掉，只留下整数部分。

`solve_24` 的约定很简单：由于有解时交回算式字符串，没有解时返回 `None`，因此调用者只要判断其返回值即可，例如下面这段代码就是它的实现。

In [3]:
def solve_24(*numbers):
    solver = Solver()
    n = RealVector('n', 4)      # 重排后的 4 个数（实数）
    op = IntVector('op', 3)     # 3 个运算符（0〜5）
    acc = RealVector('s', 3)    # 中途的计算结果

    solver.add(permutation(n, numbers))

    for x in op:
        solver.add(0 <= x, x < 6)

    def calc_op(n1, o, n2, res):
        """res == n1 (o) n2 这个约束"""
        return And(
            Implies(n2 == 0, o != 3),        # 不能除以 0
            Implies(n1 == 0, o != 5),
            Implies(o == 0, res == n1 + n2),
            Implies(o == 1, res == n1 - n2),
            Implies(o == 2, res == n1 * n2),
            Implies(o == 3, res == n1 / n2),
            Implies(o == 4, res == n2 - n1),  # 左右交换的减法
            Implies(o == 5, res == n2 / n1),  # 左右交换的除法
        )

    solver.add(calc_op(n[0], op[0], n[1], acc[0]))    # s0 = n0 op0 n1
    solver.add(calc_op(acc[0], op[1], n[2], acc[1]))  # s1 = s0 op1 n2
    solver.add(calc_op(acc[1], op[2], n[3], acc[2]))  # s2 = s1 op2 n3
    solver.add(acc[2] == 24)

    if solver.check() != sat:
        return None

    m = solver.model()
    numbers_used = [m.eval(x).as_long() for x in n]
    ops_used = [m.eval(x).as_long() for x in op]
    return output(numbers_used, ops_used)

下面挑几道题试手，拼出来的式子是否真的等于 24，用 `eval()` 算一遍就知道。

In [4]:
questions = [
    [1, 5, 5, 5],
    [3, 3, 8, 8],
    [3, 3, 7, 7],
    [1, 4, 5, 6],
    [2, 2, 2, 9],
    [2, 7, 8, 9],
    [6, 9, 9, 10],
    [1, 2, 7, 7],
    [4, 4, 10, 10],
    [2, 5, 5, 10],
]

for q in questions:
    expr = solve_24(*q)
    if expr is None:
        print(f'{q}: 无解')
    else:
        print(f'{q}: {expr} = {eval(expr):g}')

[1, 5, 5, 5]: (5 - (1 / 5)) * 5 = 24
[3, 3, 8, 8]: 8 / (3 - (8 / 3)) = 24
[3, 3, 7, 7]: ((3 / 7) + 3) * 7 = 24


[1, 4, 5, 6]: 6 / ((5 / 4) - 1) = 24


[2, 2, 2, 9]: ((2 + 9) * 2) + 2 = 24


[2, 7, 8, 9]: ((7 + 9) * 2) - 8 = 24
[6, 9, 9, 10]: (9 / (6 / 10)) + 9 = 24
[1, 2, 7, 7]: ((7 * 7) - 1) / 2 = 24
[4, 4, 10, 10]: ((10 * 10) - 4) / 4 = 24


[2, 5, 5, 10]: (5 - (2 / 10)) * 5 = 24


每道题都解出来了，不过要注意，Z3 交回的是满足条件的第一个解，因此同一道题跑两次，算式可能长得不一样。例如 `[1, 4, 5, 6]` 就有下面两种写法。

- `6 / ((5 / 4) - 1)`
- `4 / (1 - (5 / 6))`

算出来的结果都是 24。

## 用 Function 重写

排列还可以换一种存法：用 `Function()` 做成一张“编号 → 数”的对应表，由编号去查数。

```python
number_map = Function('number_map', IntSort(), RealSort())
```

读一下这个签名，它表示吃进整数、吐出实数的函数。再用 `number_map(0) == numbers[0]` 这样一条一条，把编号和具体的数钉在一起，表的内容就确定了。

使用的时候，先 `indices = IntVector('i', 4)`，再加一条 `Distinct(indices)`，逼着四个编号互不相同。编号一旦不同，从 `number_map` 里取出来的数自然就是一次重排，这与前一种写法要表达的意思完全一样。

In [5]:
def solve_24_by_function(*numbers):
    # 编号 → 数 的对应表
    number_map = Function('number_map', IntSort(), RealSort())

    solver = Solver()
    for i, v in enumerate(numbers):
        solver.add(number_map(i) == v)

    # 取出数的顺序（4 个编号互不相同）
    indices = IntVector('i', 4)
    for x in indices:
        solver.add(0 <= x, x < 4)
    solver.add(Distinct(indices))

    op = IntVector('op', 3)
    for x in op:
        solver.add(0 <= x, x < 6)

    acc = RealVector('s', 3)

    def calc_op(n1, o, n2, res):
        return And(
            Implies(n2 == 0, o != 3),
            Implies(n1 == 0, o != 5),
            Implies(o == 0, res == n1 + n2),
            Implies(o == 1, res == n1 - n2),
            Implies(o == 2, res == n1 * n2),
            Implies(o == 3, res == n1 / n2),
            Implies(o == 4, res == n2 - n1),
            Implies(o == 5, res == n2 / n1),
        )

    n = [number_map(i) for i in indices]
    solver.add(calc_op(n[0], op[0], n[1], acc[0]))
    solver.add(calc_op(acc[0], op[1], n[2], acc[1]))
    solver.add(calc_op(acc[1], op[2], n[3], acc[2]))
    solver.add(acc[2] == 24)

    if solver.check() != sat:
        return None

    m = solver.model()
    numbers_used = [m.eval(x).as_long() for x in n]
    ops_used = [m.eval(x).as_long() for x in op]
    return output(numbers_used, ops_used)


for q in questions:
    expr = solve_24_by_function(*q)
    print(f'{q}: {expr} = {eval(expr):g}')

[1, 5, 5, 5]: (5 - (1 / 5)) * 5 = 24
[3, 3, 8, 8]: 8 / (3 - (8 / 3)) = 24
[3, 3, 7, 7]: ((3 / 7) + 3) * 7 = 24


[1, 4, 5, 6]: 6 / ((5 / 4) - 1) = 24
[2, 2, 2, 9]: ((2 + 9) * 2) + 2 = 24


[2, 7, 8, 9]: ((7 + 9) * 2) - 8 = 24
[6, 9, 9, 10]: (10 / (6 / 9)) + 9 = 24


[1, 2, 7, 7]: ((7 * 7) - 1) / 2 = 24


[4, 4, 10, 10]: ((10 * 10) - 4) / 4 = 24
[2, 5, 5, 10]: (5 - (2 / 10)) * 5 = 24


答案与前面一模一样，两种写法真正的差别只在于排列怎样表示。

| | 用 `permutation()` | 用 `Function` |
|---|---|---|
| 数的重排 | 靠值的出现次数来约束 | 编号 → 数 的对应表，再加 `Distinct` |
| 有多个相同的数时 | 数出现次数，天然处理得了 | 编号不同就够，值相同也无所谓(同样天然处理得了) |
| 式子的长度 | 短 | 稍长一点 |

两条路都通。数一多、种类一杂，对应表那种写法看起来会更顺，改动起来也更清楚。

## 小结

- 四个数的重排，`permutation()`(按出现次数约束)能写，`Function` 加 `Distinct` 也能写
- 中间结果会出现分数，因此变量一律设成 `Real`
- 运算符编成 0〜5，再用 `Implies` 把“这个编号对应什么计算”逐条摆开
- 减法与除法把左右交换的形式也编上编号，其重排就不必跟着翻倍
- 解未必只有一个，Z3 交回的是满足条件的第一个，换个顺序执行就可能得到另一条式子